In [1]:
import librosa
import torch
import torch.nn.functional as F
import os
import numpy as np
from pathlib import Path
import copy # for deepcopy

import matplotlib.pyplot as plt
from IPython.display import Audio, display

from transformers import EncodecModel, AutoProcessor

from realtime_synth_ui import build_synth_ui # pip install "rtpysynth[ui] @ git+https://github.com/lonce/RTPySynth@v0.1.4"

/home/lonce/miniconda3/envs/basicaudio/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
import time

In [3]:
# for the rt synth
from realtime_synth.generators.base import BaseGenerator
from realtime_synth.utils import exp_map01
from realtime_synth_ui import build_synth_ui

# import the system demo synths just to have them on the interface
from realtime_synth.generators.sine import SineGenerator
from realtime_synth.generators.noisy_lp import NoisyLPGenerator

In [4]:
# for RNN4Control
from model.gru_audio_model import RNN, GRUModelConfig
from audioDataLoader.audio_dataset import  efficient_codes_to_latents, preprocess_latents_for_RNN # , latents_to_audio_simple,

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>Synth/Encodec Parameters</b>

In [5]:
#audiopath = data_dir / "DSBugs--busybodyFreqFactor-00.60--c-00--x-00.wav"
Kbs = 3  # Supported bandwidths are 1.5kbps (n_q = 2), 3 kbps (n_q = 4), 6 kbps (n_q = 8) and 12 kbps (n_q =16) and 24kbps (n_q=32).
buffersize = 320 #[NOTE - not tested on values other than 24000/75 - the frame length of encodec codes in samples]

g_hopsize=5
g_chunksize=10
sr=24000

g_param_labels = ["param 1", "Amp"]
g_norm_param_vals = [.5, .5]  #for the synth whose paramters can be different than those it sends to the NN
g_init_cond=[.5]

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>RNN Parameters</b>

In [6]:
# Where is the checkpoint directory?
run_directory = str(Path('./output/20250828_173834_encodectest_DSWind')) # 'Path to the directory of the saved run.'
run_directory = str(Path('./output/20250827_164925_bees_good')) # 'Path to the directory of the saved run.'
run_directory = str(Path('./output/20250829_113424_encodectest_ChirpPattern')) # 'Path to the directory of the saved run.' 

g_top_n = 32 #'Sample from the top N most likely outputs.'
g_temperature = 1 #'Controls the randomness of predictions.'

frame_rate=75
device='cpu'

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>RNN CLASS</b>

In [7]:
class RNNGenerator():
    def __init__(self, checkpoint_path, model_config, data_config, enc_model, chunksize, hopsize, init_cond, top_n, temperature) : 
        #self.chkpt = chkpt

        self.clamp_val = data_config.clamp_val # need this to map between encodec latents and model input ranges

        self.model = RNN(model_config).to(device)
        checkpoint = torch.load(checkpoint_path, map_location=device)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.model.eval()

        self.enc_model=enc_model

        self.codebook_size = self.model.config.codebook_size
        self.n_q = self.model.config.n_q
        self.cond_size = self.model.config.cond_size
    

        self.chunksize=chunksize
        self.hopsize=hopsize

        self.top_n=top_n
        self.temperature=temperature

        #state between call to generate steps
        self.hidden = None # updated on every sequence step in warmup and in run_inference
        
        self.current_latent = self.warmup_rnn(init_cond) # initializes self.current_latent

        #initialize the "current_code_chunk" so we have the history window ready for the first and following getNextCodeChunk calls
        self.current_code_chunk = self.run_inference(init_cond, self.chunksize) #TIME should be the last dimension
        

    def warmup_rnn(self, init_cond, warmup_len=10) :
        sd=.33 # to create data in [-1,1
        warmup_latents = warmup_tensor = torch.clamp(torch.randn(warmup_len, 128) * sd, -3*sd, 3*sd)  

            # Ensure warmup_latents is on the right device and has correct shape
        warmup_latents = warmup_latents.to(device)  # Shape: (warmup_len, 128)
        #warmup_len = warmup_latents.shape[0]
    
        # Handle conditioning
        if self.cond_size > 0 and init_cond is not None:
            init_cond_tensor = torch.tensor(init_cond)
            # Use first conditioning vector for entire warmup
            first_cond_vec = init_cond_tensor.unsqueeze(0).repeat(warmup_len, 1).to(device)  # (warmup_len, cond_size)
            warmup_full_input = torch.cat([warmup_latents, first_cond_vec], dim=-1)  # (warmup_len, 128 + cond_size)
        else:
            # No conditioning
            warmup_full_input = warmup_latents  # (warmup_len, 128)
        
        self.hidden = self.model.init_hidden(batch_size=1)
        # This propogates hidden state, but doesn't bring the output back down for the next input, using the warmup vectors instead.
        for i in range(len(warmup_full_input)):
            _, self.hidden = self.model(warmup_full_input[i].unsqueeze(0), self.hidden, batch_size=1)
        
        # Get the last latent for starting generation
        return warmup_latents[-1].unsqueeze(0)  # (1, 128)


        
    #return the next hopsize codes
    def run_inference(self, params, generation_length) :
        generated_codes = []
        with torch.no_grad():
            for i in range(generation_length):
                # Handle conditioning for this step
                if self.cond_size > 0 and params is not None:
                    current_cond_vec = torch.tensor(params).unsqueeze(0).to(device)  # (1, cond_size)
                    next_input_full = torch.cat([self.current_latent, current_cond_vec], dim=-1)  # (1, 128 + cond_size)
                else:
                    # No conditioning
                    next_input_full = self.current_latent  # (1, 128)
    
                logits_list, self.hidden = self.model(next_input_full, self.hidden, batch_size=1)
    
                # Transform outputs to next input using the extracted function
                self.current_latent, sampled_codes = self.transform_outputs_to_inputs(logits_list)
                generated_codes.append(sampled_codes)
        return np.array(generated_codes).T.tolist()  # make time the last dimension for the encodec decoder
        
        
    def getNextCodeChunk(self, params, generation_length) :
        # Generate new hopsize codes
        hopcodes = self.run_inference(params, generation_length)
        # push them on to the end of current_code_chunk
        self.current_code_chunk = [(row + b_row)[generation_length:] for row, b_row in zip(self.current_code_chunk, hopcodes)]
        # return the new chunk
        return self.current_code_chunk     #and return it
        
    def transform_outputs_to_inputs(self, logits_list):
        """
        Transform model outputs (logits) into the next input (128D latent).
        
        Args:
            logits_list: List of logit tensors, one per quantizer
            encodec_model: EnCodec model for code->latent conversion
            clamp_val (float) - clamp latents (produced by encodec token decoding) in [-clamp_val, clampval], the map to [-1,1] for input to model next step
            top_n: Number of top predictions to sample from
            temperature: Sampling temperature
            codebook_size: Size of each codebook
            n_q: Number of quantizers
        
        Returns:
            torch.Tensor: Next input latent of shape (1, 128)
        """
        device = logits_list[0].device
        self.enc_model.to(device)
        sampled_codes = []
        
        for j in range(self.n_q):
            # Apply temperature and get top-k
            logits_j = logits_list[j].div(self.temperature).squeeze()  # (codebook_size,)
            top_n_logits, top_n_indices = torch.topk(logits_j, self.top_n)
            top_n_probs = F.softmax(top_n_logits, dim=-1)
            
            # Sample from top-k
            try:
                sampled_relative_idx = torch.multinomial(top_n_probs, 1).squeeze()
                sampled_code = top_n_indices[sampled_relative_idx]
                sampled_codes.append(sampled_code.item())
            except Exception as e:
                print(f"Sampling error for quantizer {j}: {e}")
                # Fallback to random sampling
                sampled_codes.append(torch.randint(0, self.codebook_size, (1,)).item())
        
        # Convert sampled codes back to latent - CREATE TENSOR ON CORRECT DEVICE
        codes_tensor = torch.tensor(sampled_codes, device=device).unsqueeze(0).unsqueeze(-1)  # (1, n_q, 1)
        next_latent = efficient_codes_to_latents(self.enc_model, codes_tensor).squeeze(0).squeeze(-1).unsqueeze(0)  # (1, 128)
        next_latent = preprocess_latents_for_RNN(next_latent, self.clamp_val)
        
        return next_latent, sampled_codes    

In [8]:
# load the encoder model + processor (for pre-processing the audio)
#####################################################################
enc_model = EncodecModel.from_pretrained("facebook/encodec_24khz")
enc_model.eval()
enc_model.config.target_bandwidths = [Kbs] # Supported bandwidths are 1.5kbps (n_q = 2), 3 kbps (n_q = 4), 6 kbps (n_q = 8) and 12 kbps (n_q =16) and 24kbps (n_q=32).
#processor = AutoProcessor.from_pretrained("facebook/encodec_24khz", use_fast=False)
enc_model.device

device(type='cpu')

In [9]:
# load the RNN model 
#####################################################################
config_path = os.path.join(run_directory, "config.pt")
checkpoint_path = os.path.join(run_directory, "checkpoints", "last_checkpoint.pt") #   # "last_checkpoint.pt") # "checkpoint_40.pt") # 

assert os.path.exists(run_directory), f"Run directory not found: {run_directory}"
assert os.path.exists(config_path), f"Config file not found: {config_path}"
assert os.path.exists(checkpoint_path), f"Checkpoint file not found: {checkpoint_path}"

saved_configs = torch.load(config_path, weights_only=False)
model_config = saved_configs["model_config"]
data_config = saved_configs["data_config"]

rnngen = RNNGenerator(checkpoint_path, model_config, data_config, enc_model, g_chunksize, g_hopsize, g_init_cond, g_top_n, g_temperature)
print("Model successfully loaded from checkpoint.")
print(f"Using device = {device}")
rnngen.clamp_val 

Model successfully loaded from checkpoint.
Using device = cpu


15

In [10]:
print(f"rnngen.current_code_chunk is {rnngen.current_code_chunk}")
rnngen.current_latent

rnngen.current_code_chunk is [[62, 62, 62, 62, 62, 62, 62, 62, 62, 62], [518, 424, 518, 518, 518, 518, 518, 518, 518, 518], [822, 36, 786, 786, 786, 678, 786, 786, 786, 786], [944, 673, 673, 673, 673, 673, 673, 673, 673, 673]]


tensor([[-0.0446,  0.5679, -0.2358, -0.1066, -0.0587,  0.3533, -0.0910, -0.0156,
         -0.1901,  0.1188,  0.0516,  0.0238,  0.0724, -0.3134, -0.0744, -0.3059,
          0.0555,  0.1209, -0.1214,  0.0237, -0.6519, -0.3298, -0.2406,  0.0363,
          0.3292, -0.0317,  0.3986, -0.2639, -0.2421, -0.1589, -0.3544,  0.2378,
          0.1923, -0.1047,  0.1622, -0.2124, -0.2323, -0.4446,  0.0317,  0.1264,
          0.3603,  0.1827,  0.6444,  0.1726,  0.0631, -0.1584, -0.2338,  0.1069,
          0.1338, -0.2479, -0.1528, -0.3278,  0.0168,  0.0580, -0.1967,  0.0127,
         -0.0435, -0.2819, -0.0457,  0.0230, -0.4198, -0.2879,  0.0051, -0.1002,
         -0.3613, -0.0602,  0.0333, -0.1726,  0.0327,  0.1826, -0.0627, -0.3527,
         -0.0353, -0.1531, -0.2105, -0.6778,  0.2261, -0.1417, -0.2530,  0.0440,
         -0.0869, -0.3390, -0.4330,  0.1667,  0.0987, -0.1185,  0.0261, -0.0376,
          0.2139, -0.1070,  0.6792,  0.3547,  0.0429,  0.1684, -0.4192,  0.4077,
          0.0765,  0.7696,  

In [11]:
newfoo = rnngen.getNextCodeChunk(g_init_cond, g_hopsize)
print(f"rnngen.current_code_chunk is {rnngen.current_code_chunk}")
rnngen.current_latent

rnngen.current_code_chunk is [[62, 62, 62, 62, 62, 807, 929, 356, 43, 259], [518, 518, 518, 518, 518, 518, 315, 167, 653, 642], [678, 786, 786, 786, 786, 786, 510, 908, 316, 366], [673, 673, 673, 673, 673, 673, 942, 610, 54, 316]]


tensor([[-5.9721e-02,  9.6907e-01, -2.9217e-01, -7.0925e-02,  5.9612e-02,
          6.9854e-01, -2.5486e-01,  1.3474e-01, -4.4099e-02,  4.1788e-01,
         -3.1335e-02, -7.3832e-03,  1.9905e-01, -1.2594e-01,  8.5862e-02,
         -7.9073e-01, -2.4587e-01,  2.6201e-01,  3.5646e-02, -1.5730e-01,
         -1.0000e+00, -1.6217e-01, -3.8912e-02, -7.8146e-02,  3.1982e-01,
         -3.9477e-02,  8.7347e-01, -4.4462e-01, -5.4859e-01, -1.4726e-01,
         -8.1908e-01,  1.4920e-01,  1.8088e-01, -5.7887e-01,  2.7981e-01,
         -3.5382e-01, -4.0874e-02, -8.8746e-01,  2.7668e-04,  4.2452e-01,
          2.3831e-01,  1.6192e-01,  1.0000e+00,  1.5247e-01, -1.1584e-02,
          6.7383e-02, -2.4366e-01,  5.1106e-01, -7.7901e-02, -3.1236e-01,
         -2.3506e-01, -5.6220e-01,  2.4442e-01,  4.9830e-01, -4.3726e-01,
          2.0548e-01, -3.1660e-01, -5.3286e-01, -1.2242e-01,  1.4327e-01,
         -5.4537e-01, -3.5275e-01,  1.0832e-01, -1.3007e-01, -7.1548e-01,
         -3.4188e-01,  2.2095e-02, -9.

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>CodecSynth</b>

In [12]:
# def prepend_encodec_seeding_segment(x: torch.Tensor, n: int) -> torch.Tensor:
#     """
#     Return a new tensor with the first `n` frames copied to the *beginning*.
#     x can be [B, C, Q, T] (time last); preserves all leading dims.
#     Result shape: [..., T+n]
#     """
#     n = int(max(0, n))
#     if n == 0:
#         return x
#     T = x.size(-1)
#     n = min(n, T)             # clamp
#     head = x[..., :n]         # keep all leading dims
#     return torch.cat([head, x], dim=-1)

In [13]:
class MyEncodecPlayer(BaseGenerator):
    # normalized params in [0,1]
    param_labels = g_param_labels

    # -----------------------
    def __init__(self, rnngen, buffersize, init_norm_params=None):
        super().__init__(init_norm_params or [0.5, 0.6])  # defaults
        self.set_params(self.norm_params)  # initialize semantic values

        self.rnngen = rnngen
        self.cond_size=rnngen.cond_size

        self.chunksizeframes=g_chunksize # decode this many frames each time
        self.framehopsize=g_hopsize # decode a new chunk every framehopsize
        self.nextendframe=self.framehopsize
        

        self.buffersize=buffersize
        self.nextsample = 0
        self.framesizesamples=sr//frame_rate # 75 (independent of g_chunksize) because encoder is 75 frames per second

        self.currentchunkframe=0 #nth frame in the chunk of audio we are playing

        self.seeding_len = self.chunksizeframes-self.framehopsize  # how many time steps to duplicate/append
        self.genaudioframe=0 #mth frame weve generated in total (considering warm up already generated)


        # now grab first chunk of audio (This is exactly the getNextAudioHop function, but we can't call it in init()!
        with torch.inference_mode():                                                                                                                      #scales, padding length[0]
            # print(f" in init, passing getNextCodeChunk this: {self.norm_params[:self.cond_size], self.framehopsize}")
            FOO = rnngen.getNextCodeChunk(self.norm_params[:self.cond_size], self.framehopsize)
            # bar = torch.tensor(FOO).unsqueeze(0).unsqueeze(0)
            # print(f"the return value, when cast as a tensor and unsqueezed, has  this shape: {bar.shape} for passing to enc_model.decode")
            self.thisaudioseq = enc_model.decode(torch.tensor(FOO).unsqueeze(0).unsqueeze(0), [None], None)[0]
        self.thisaudioseq=self.thisaudioseq[0, 0].detach().numpy()
        self.thisaudioseq=self.thisaudioseq[-self.framehopsize*self.framesizesamples:]
        
        
        self.nextaudioseq=None  #compute ahead of time, same till ready to use

        self._callrecord=f""
        self._decodetime=0

    # -----------------------
    def getNextAudioHop(self):
        self.genaudioframe = self.genaudioframe+self.framehopsize

        self._callrecord = self._callrecord + f";(start: {self.genaudioframe}, end: {self.genaudioframe  + self.chunksizeframes})"
            
        #decode a whole chunk 
        start_time = time.monotonic()
        with torch.inference_mode():                                                                                                              #scales, padding length[0]
            nextseq = enc_model.decode(torch.tensor(rnngen.getNextCodeChunk(self.norm_params[:self.cond_size], self.framehopsize)).unsqueeze(0).unsqueeze(0), [None], None)[0]
        nextseq=nextseq[0, 0].detach().numpy()
        self._decodetime = self._decodetime  + time.monotonic() - start_time
        
        #but take only the hopsize of audio that we need
        return nextseq[-self.framehopsize*self.framesizesamples:]
        
    # -----------------------   
    def set_params(self, norm_params):
        super().set_params(norm_params)
        # Map [0,1] → semantic values
        self.freq = float(exp_map01(self.norm_params[0], 20.0, 2000.0))  # exponential Hz
        self.amp  = float(self.norm_params[1])                           # linear gain 0..1 

    # -----------------------
    def generate(self, frames, sr):
       
        assert frames == self.buffersize, "ooh, you're in trouble if frames requested is different than the buffer size."

        if self.amp <= 0.0 or self.freq <= 0.0 or self.freq < 0.0:
            return np.zeros(self.buffersize, dtype=np.float32)
            
        endsamp=self.nextsample+self.buffersize
        y = self.thisaudioseq[self.nextsample:endsamp]
        self.nextsample=endsamp

        #This is how to get error reports out of the generation thread! After Playing, int he following cell call:
        #      print(getattr(synth.gen, "_last_error", None))
        if self.currentchunkframe==0 :
            try:
                self.nextaudioseq = self.getNextAudioHop()   # fast, non-blocking work only
            except Exception as e:
                self._last_error = repr(e)
                return np.zeros(frames, dtype=np.float32)

        try:    
            self.currentchunkframe = self.currentchunkframe+1
            
            if self.currentchunkframe==self.framehopsize :
                 self.thisaudioseq = self.nextaudioseq # it should be waiting for us
                 self.currentchunkframe = 0
                 self.nextsample = 0
        except Exception as e:
            self._last_error = repr(e)
             
        return self.amp*y.astype(np.float32)

    # -----------------------
    def formatted_readouts(self):
        # Optional: pretty labels shown next to sliders
        return [f"{self.param_labels[0]}: {self.freq:7.2f} Hz",
                f"{self.param_labels[1]}: {self.amp:.3f}"]
        

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>ON LINE, Realtime TEST</b>

In [14]:
GENS = {
    "MyEncodecPlayer": lambda: MyEncodecPlayer(rnngen, buffersize, g_norm_param_vals),
    "Sine": SineGenerator,            # defined in the default system
    "Noisy LP": NoisyLPGenerator,     # defined in the default system
}
print(f"Will use samplerate = {sr} and blocksize = {buffersize}")
synth, ui = build_synth_ui(GENS, samplerate=sr, blocksize=buffersize, channels=1)

Will use samplerate = 24000 and blocksize = 320


HTML(value='')

In [15]:
print(getattr(synth.gen, "_last_error", None))

None


In [16]:
print(getattr(synth.gen, "_callrecord ", None))

None


In [17]:
print(getattr(synth.gen, "__decodetime", None))

None


<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>OFF LINE TEST</b>

In [18]:
# just checking, not passed to to the synth which takes classes
foo=MyEncodecPlayer(rnngen, buffersize, g_norm_param_vals)
foo.thisaudioseq
len(foo.thisaudioseq)
#display(Audio(foo.thisaudioseq, rate=sr))

1600

In [19]:
# chunk=foo.thisaudioseq
# for i in range(0,1) :
#     foo.getNextAudioHop()
#     chunk = np.concatenate((chunk, foo.thisaudioseq), axis=0) 
# len(chunk)

In [20]:
c=[]
secs=5
hops=int(secs*frame_rate/g_hopsize)


for hopnum in range(0,hops) :
    c=  np.concatenate((c , foo.thisaudioseq), axis=0) 
    foo.thisaudioseq=foo.getNextAudioHop()
print(f"time spent decoding = {foo._decodetime:.2f}")


time spent decoding = 1.04


In [21]:
display(Audio(c, rate=sr))

In [22]:
print(f"{foo._callrecord}")

;(start: 5, end: 15);(start: 10, end: 20);(start: 15, end: 25);(start: 20, end: 30);(start: 25, end: 35);(start: 30, end: 40);(start: 35, end: 45);(start: 40, end: 50);(start: 45, end: 55);(start: 50, end: 60);(start: 55, end: 65);(start: 60, end: 70);(start: 65, end: 75);(start: 70, end: 80);(start: 75, end: 85);(start: 80, end: 90);(start: 85, end: 95);(start: 90, end: 100);(start: 95, end: 105);(start: 100, end: 110);(start: 105, end: 115);(start: 110, end: 120);(start: 115, end: 125);(start: 120, end: 130);(start: 125, end: 135);(start: 130, end: 140);(start: 135, end: 145);(start: 140, end: 150);(start: 145, end: 155);(start: 150, end: 160);(start: 155, end: 165);(start: 160, end: 170);(start: 165, end: 175);(start: 170, end: 180);(start: 175, end: 185);(start: 180, end: 190);(start: 185, end: 195);(start: 190, end: 200);(start: 195, end: 205);(start: 200, end: 210);(start: 205, end: 215);(start: 210, end: 220);(start: 215, end: 225);(start: 220, end: 230);(start: 225, end: 235);(

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>OFF LINE TEST 2 - calling generate</b>

In [23]:
bar= MyEncodecPlayer(rnngen, buffersize, g_norm_param_vals)

In [24]:
audio_out=[]
#bar._decodetime=0
for buffernum in range(0,1375) :
    audio_out =  np.concatenate((audio_out, bar.generate(buffersize, sr)), axis=0) 
print(f"time spent decoding = {bar._decodetime:.2f}")
display(Audio(audio_out, rate=sr))    

time spent decoding = 4.00
